# Gradient descent: hands-on (optional)
**LLMs, Animated · Lesson P2.2**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*

Only `numpy` and `matplotlib` are needed (both pre-installed on Colab).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("ready")

## 1 · The slope of a loss
Our loss has one parameter $\theta$:

$$L(\theta) = (\theta - 3)^2 + 1$$

Its minimum is at $\theta = 3$. Implement its derivative $\frac{dL}{d\theta}$.
*(Hint: the chain rule gives $2(\theta - 3)$.)*

In [ ]:
def loss(theta):
    return (theta - 3) ** 2 + 1

In [ ]:
def grad(theta):
    # TODO: return dL/dθ for L(θ) = (θ - 3)² + 1
    raise NotImplementedError("Exercise 1")

**Check 1.** We also compare against a *numerical* derivative, a trick worth remembering for debugging gradients.

In [ ]:
assert grad(3) == 0, "the slope at the minimum should be 0"
assert grad(5) == 4
numeric = (loss(1.0 + 1e-5) - loss(1.0 - 1e-5)) / 2e-5   # central difference
assert abs(grad(1.0) - numeric) < 1e-4
print("✓ exercise 1")

## 2 · The update loop
Implement gradient descent:

$$\theta \leftarrow \theta - \eta \, \frac{dL}{d\theta}$$

Return the list of every $\theta$ visited, **including** the starting value (so `steps` updates give `steps + 1` values).

In [ ]:
def gradient_descent(grad_fn, theta0, lr, steps):
    thetas = [theta0]
    # TODO: repeat `steps` times: update theta with the rule above and append it
    raise NotImplementedError("Exercise 2")

**Check 2**

In [ ]:
hist = gradient_descent(grad, theta0=-2.0, lr=0.1, steps=100)
assert len(hist) == 101
assert abs(hist[-1] - 3) < 1e-6, hist[-1]
assert all(loss(a) >= loss(b) for a, b in zip(hist, hist[1:])), "loss should never go up with lr=0.1"
print("✓ exercise 2, final θ =", round(hist[-1], 6))

## 3 · Feel the learning rate
Run the cell to plot the loss over 30 steps for four learning rates. Then answer in the TODO cell.

For this loss the update is $\theta - 3 \leftarrow (1 - 2\eta)(\theta - 3)$, so the distance to the minimum is multiplied by $(1-2\eta)$ every step.

In [ ]:
rates = [0.01, 0.1, 0.9, 1.05]
fig, ax = plt.subplots(figsize=(7, 4))
for lr in rates:
    h = gradient_descent(grad, theta0=-2.0, lr=lr, steps=30)
    ax.plot([loss(t) for t in h], marker="o", ms=3, label=f"η = {lr}")
ax.set_yscale("log"); ax.set_xlabel("step"); ax.set_ylabel("loss (log scale)")
ax.legend(); ax.set_title("Same problem, four learning rates")
plt.show()

In [ ]:
# TODO: fill in one learning rate from `rates` for each behaviour
too_slow_lr = None      # converges, but barely moves in 30 steps
oscillating_lr = None   # jumps back and forth across the minimum, yet still converges
diverging_lr = None     # the loss grows without bound

**Check 3**

In [ ]:
assert too_slow_lr == 0.01 and oscillating_lr == 0.9 and diverging_lr == 1.05
h = gradient_descent(grad, -2.0, oscillating_lr, 6)
signs = np.sign(np.array(h) - 3)
assert (signs[1:] != signs[:-1]).all(), "with η = 0.9 θ should land on alternating sides of 3"
print("✓ exercise 3")

## 4 · Two parameters: the zig-zag
Real models have many parameters. Take two, with a long, thin bowl:

$$L(x, y) = x^2 + 10\,y^2$$

The **gradient** is the vector of partial derivatives $\nabla L = \left(\frac{\partial L}{\partial x}, \frac{\partial L}{\partial y}\right)$. Implement it, run gradient descent, and look at the path over the contour lines.

In [ ]:
def loss2d(p):
    x, y = p
    return x ** 2 + 10 * y ** 2

In [ ]:
def grad2d(p):
    x, y = p
    # TODO: return np.array([∂L/∂x, ∂L/∂y])
    raise NotImplementedError("Exercise 4a")

def gd2d(p0, lr, steps):
    p = np.array(p0, dtype=float)
    path = [p.copy()]
    # TODO: same loop as before, but p is now a vector
    raise NotImplementedError("Exercise 4b")

In [ ]:
path = gd2d([-4.0, 1.0], lr=0.085, steps=25)
xs, ys = np.meshgrid(np.linspace(-4.5, 1.5, 200), np.linspace(-1.6, 1.6, 200))
plt.figure(figsize=(7, 4))
plt.contour(xs, ys, xs ** 2 + 10 * ys ** 2, levels=[0.5, 2, 5, 10, 18, 26], colors="gray", linewidths=0.8)
plt.plot(path[:, 0], path[:, 1], "o-", color="tab:orange", ms=3)
plt.gca().set_aspect("equal"); plt.title("Gradient descent zig-zags across the narrow valley")
plt.show()

Notice: each step points **straight downhill**, perpendicular to the contour line, *not* toward the minimum. The steep $y$ direction makes it bounce while the gentle $x$ direction crawls. (Optimizers like Adam, lesson M2.3, fix exactly this.)

**Question:** the $y$-coordinate is multiplied by $(1 - 20\eta)$ each step. What is the largest learning rate that does *not* diverge?

In [ ]:
max_stable_lr = None  # TODO: |1 - 20η| < 1  ⇒  η < ?

**Check 4**

In [ ]:
assert np.allclose(grad2d([1.0, 1.0]), [2, 20])
assert loss2d(path[-1]) < 0.01 * loss2d(path[0])
assert abs(max_stable_lr - 0.1) < 1e-9
assert not np.isfinite(loss2d(gd2d([-4.0, 1.0], 0.11, 400)[-1])) or loss2d(gd2d([-4.0, 1.0], 0.11, 400)[-1]) > 1e6
print("✓ exercise 4")

## 5 · Mini-batch SGD on real-ish data
Fit a line $\hat y = w x + b$ to 1,000 noisy points with the **mean squared error** loss

$$L(w, b) = \frac{1}{n}\sum_i (w x_i + b - y_i)^2, \qquad \frac{\partial L}{\partial w} = \frac{2}{n}\sum_i (w x_i + b - y_i)\,x_i, \qquad \frac{\partial L}{\partial b} = \frac{2}{n}\sum_i (w x_i + b - y_i)$$

Instead of the full dataset, each step uses a random **mini-batch** of 32 points. That's exactly how LLMs are trained (with batches of millions of tokens).

In [ ]:
n = 1000
X = np.random.uniform(-3, 3, n)
Y = 2.0 * X + 1.0 + np.random.normal(0, 0.5, n)   # true w = 2, b = 1

In [ ]:
def mse_grad(w, b, xb, yb):
    # TODO: return (dL/dw, dL/db) on the mini-batch xb, yb
    raise NotImplementedError("Exercise 5a")

def train(lr=0.05, epochs=20, batch=32):
    w, b = 0.0, 0.0
    losses = []
    for _ in range(epochs):
        idx = np.random.permutation(n)          # shuffle once per epoch
        for s in range(0, n, batch):
            j = idx[s:s + batch]
            # TODO: compute the mini-batch gradient and update w and b
            raise NotImplementedError("Exercise 5b")
        losses.append(np.mean((w * X + b - Y) ** 2))  # full-data loss after each epoch
    return w, b, losses

**Check 5**

In [ ]:
w, b, losses = train()
print(f"w = {w:.3f}, b = {b:.3f}")
assert abs(w - 2) < 0.1 and abs(b - 1) < 0.1
assert losses[-1] < 0.35
plt.plot(losses, marker="o"); plt.xlabel("epoch"); plt.ylabel("MSE"); plt.title("Mini-batch SGD"); plt.show()
print("✓ exercise 5")

## Recap
- The derivative/gradient tells you which way is **uphill**; step the other way.
- The learning rate $\eta$ sets the step size: too small crawls, too big oscillates or diverges.
- With many parameters, gradient descent can zig-zag in narrow valleys.
- Mini-batches give noisy but cheap gradient estimates. This is how every LLM is trained.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def grad(theta):
    return 2 * (theta - 3)

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def gradient_descent(grad_fn, theta0, lr, steps):
    thetas = [theta0]
    theta = theta0
    for _ in range(steps):
        theta = theta - lr * grad_fn(theta)
        thetas.append(theta)
    return thetas

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
too_slow_lr = 0.01
oscillating_lr = 0.9
diverging_lr = 1.05

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def grad2d(p):
    x, y = p
    return np.array([2 * x, 20 * y])

def gd2d(p0, lr, steps):
    p = np.array(p0, dtype=float)
    path = [p.copy()]
    for _ in range(steps):
        p = p - lr * grad2d(p)
        path.append(p.copy())
    return np.array(path)

max_stable_lr = 0.1

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def mse_grad(w, b, xb, yb):
    err = w * xb + b - yb
    return 2 * np.mean(err * xb), 2 * np.mean(err)

def train(lr=0.05, epochs=20, batch=32):
    w, b = 0.0, 0.0
    losses = []
    for _ in range(epochs):
        idx = np.random.permutation(n)
        for s in range(0, n, batch):
            j = idx[s:s + batch]
            dw, db = mse_grad(w, b, X[j], Y[j])
            w, b = w - lr * dw, b - lr * db
        losses.append(np.mean((w * X + b - Y) ** 2))
    return w, b, losses